In [1]:
import pandas as pd
import numpy as np
import json
import jsonlines

In [11]:
num_examples_per_label = 2

In [12]:
ICL_PROMPT = """
PROMPT COMES HERE
"""

In [8]:
test_df = pd.read_csv("../data/test/full_test.csv")
test_df

,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label
0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0
1,ar89pm,Anybody hate spring/summer?,".....And even so, when depression is not too s...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}","Anybody hate springsummer. And even so, when d...",Comorbid (Depression + Anxiety),1,1
2,5g4v1n,So I'm staring at this tablet of Lexapro...,...and I'm not sure what to do. I guess I don'...,"[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",So Im staring at this tablet of Lexapro. and I...,Comorbid (Depression + Anxiety),1,1
3,9x8h3g,"Found a poem sort of thing, very emo (ha ha ha...","""I am delicate and bitter. I am sweet on the o...","[1, 0]",{'depressive_disorder'},"Found a poem sort of thing, very emo ha ha ha ...",Depression,1,0
4,es440i,You don't get it,"""It gets better"" ""Stop thinking about it"" ""get...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",You dont get it. It gets better Stop thinking ...,Comorbid (Depression + Anxiety),1,1
...,...,...,...,...,...,...,...,...,...
2867,lm7uix,My life just feels like this one big party Im ...,You know the feeling. Where youre just so over...,"[1, 0]",{'depressive_disorder'},My life just feels like this one big party Im ...,Depression,1,0
2868,ac1aj9,Just move away from your parents!,You probably want to hear my sob story as much...,"[0, 0]",{'control_group'},Just move away from your parents. You probably...,Normal,0,0
2869,hng1f5,That feeling when...,"You wake up, and that itself just brings you s...","[0, 0]",{'control_group'},"That feeling when. You wake up, and that itsel...",Normal,0,0
2870,4pfk9x,Life just keeps getting more complicated,"You would think after years of bad breaks, som...","[0, 0]",{'control_group'},Life just keeps getting more complicated. You ...,Normal,0,0


In [10]:
silver_df = pd.read_csv("../data/silver_data/silver_labels_gpt_3.5_turbo.csv")
silver_df

,index,text,subreddit,author,system_description,prompt,gpt_prediction,Mental Health Disorder,Name of Mental Health Disorder,DSM5 Rationale,silver_label,id,depression_label,anxiety_label,multilabel_clf_label,multiclass_clf_label
0,11,"i woke up very early, 2 am. i just came out of...",ForeverAlone,SixViking,You are a Psychology professor working in the ...,"Reddit Post: ""i woke up very early, 2 am. i ju...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Major Depressive Disorder (MDD),The language used in the post indicates severa...,Depression,4JEVyZ,1,0,"[1, 0]",Depression
1,16,ive been trying my damndest to improve myself ...,depression,SixViking,You are a Psychology professor working in the ...,"Reddit Post: ""ive been trying my damndest to i...",Mental Health Disorder: Yes\n\nName of Mental ...,Yes,Major Depressive Disorder (MDD),The language used in the Reddit post indicates...,Depression,3jkFBr,1,0,"[1, 0]",Depression
2,17,he lives on the other side of the country and ...,SuicideWatch,s0laris0,You are a Psychology professor working in the ...,"Reddit Post: ""he lives on the other side of th...",Mental Health Disorder: Yes\n\nName of Mental ...,Yes,Depression with suicidal ideation,The language used in the Reddit post suggests ...,Depression,4DZQ7p,1,0,"[1, 0]",Depression
3,33,i work in retail and lately ive had some shit ...,bipolar,s0laris0,You are a Psychology professor working in the ...,"Reddit Post: ""i work in retail and lately ive ...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Anxiety Disorder,The language used in the post suggests the pre...,Anxiety,FSr2Vx,0,1,"[0, 1]",Anxiety
4,37,"i had my first one in november 2020, another 2...",Epilepsy,s0laris0,You are a Psychology professor working in the ...,"Reddit Post: ""i had my first one in november 2...",Mental Health Disorder: Yes\nName of Mental He...,Yes,Generalized Anxiety Disorder (GAD),The language used in the Reddit post indicates...,Anxiety,42FfcW,0,1,"[0, 1]",Anxiety
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7662,22138,"so far i have yew as a wizardfreelancer, magno...",bravelydefault,Cyndikate,You are a Psychology professor working in the ...,"Reddit Post: ""so far i have yew as a wizardfre...",Mental Health Disorder: No\n\nDSM5 Rationale: ...,No,NaN,The language used in the Reddit post does not ...,Normal,3eGDUP,0,0,"[0, 0]",Normal
7663,3614,i recently just purchased a rx200s mod with a ...,electronic_cigarette,zackcheese7,You are a Psychology professor working in the ...,"Reddit Post: ""i recently just purchased a rx20...",Mental Health Disorder: No\n\nDSM5 Rationale: ...,No,NaN,The language used in the Reddit post does not ...,Normal,48oGZg,0,0,"[0, 0]",Normal
7664,6592,"according to the research ive done online, see...",Skincare_Addiction,see_elle,You are a Psychology professor working in the ...,"Reddit Post: ""according to the research ive do...",Mental Health Disorder: No\n\nDSM5 Rationale: ...,No,NaN,The language used in the given Reddit post doe...,Normal,VvEkU9,0,0,"[0, 0]",Normal
7665,3582,stepped on the scale this am. 151.6. well it c...,xxketo,whymynamemeansgrace,You are a Psychology professor working in the ...,"Reddit Post: ""stepped on the scale this am. 15...",Mental Health Disorder: No\n\nDSM5 Rationale: ...,No,NaN,"Based on the language used in the Reddit post,...",Normal,35i6P5,0,0,"[0, 0]",Normal


In [2]:
with open("../data/mappings/semantic-similarity-depression.json") as f:
    depression_data = json.load(f)
f.close()
len(depression_data)

2872

In [5]:
examples_dict = dict()
for key, value in depression_data.items():
    if key not in examples_dict:
        examples_dict[key] = 

ar89pm <class 'list'> 30


In [15]:
example_keys = [val[0] for val in value[:3]]
example_keys

['kUo2W9', '448UbJ', 'Y3AmZR']

In [20]:
silver_df[silver_df['id'].isin(example_keys)]['text']

113     i live in small town south dakota and i absolu...
2142    i was diagnosed with clinical depression about...
2149    i just have to accept that im depressed and th...
Name: text, dtype: object

In [2]:
import pandas as pd
from datasets import Dataset as hf_dataset

In [3]:
prompt_data = pd.read_csv("../data/test/full_test.csv")
prompt_data

,id,title,selftext,multilabel_clf_label,disorder,text,multiclass_clf_label,depression_label,anxiety_label
0,8es6qn,I cut myself for the first time in a year toda...,... and hated that I still loved it. The burni...,"[1, 0]",{'depressive_disorder'},I cut myself for the first time in a year toda...,Depression,1,0
1,ar89pm,Anybody hate spring/summer?,".....And even so, when depression is not too s...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}","Anybody hate springsummer. And even so, when d...",Comorbid (Depression + Anxiety),1,1
2,5g4v1n,So I'm staring at this tablet of Lexapro...,...and I'm not sure what to do. I guess I don'...,"[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",So Im staring at this tablet of Lexapro. and I...,Comorbid (Depression + Anxiety),1,1
3,9x8h3g,"Found a poem sort of thing, very emo (ha ha ha...","""I am delicate and bitter. I am sweet on the o...","[1, 0]",{'depressive_disorder'},"Found a poem sort of thing, very emo ha ha ha ...",Depression,1,0
4,es440i,You don't get it,"""It gets better"" ""Stop thinking about it"" ""get...","[1, 1]","{'depressive_disorder', 'anxiety_disorder'}",You dont get it. It gets better Stop thinking ...,Comorbid (Depression + Anxiety),1,1
...,...,...,...,...,...,...,...,...,...
2867,lm7uix,My life just feels like this one big party Im ...,You know the feeling. Where youre just so over...,"[1, 0]",{'depressive_disorder'},My life just feels like this one big party Im ...,Depression,1,0
2868,ac1aj9,Just move away from your parents!,You probably want to hear my sob story as much...,"[0, 0]",{'control_group'},Just move away from your parents. You probably...,Normal,0,0
2869,hng1f5,That feeling when...,"You wake up, and that itself just brings you s...","[0, 0]",{'control_group'},"That feeling when. You wake up, and that itsel...",Normal,0,0
2870,4pfk9x,Life just keeps getting more complicated,"You would think after years of bad breaks, som...","[0, 0]",{'control_group'},Life just keeps getting more complicated. You ...,Normal,0,0


In [5]:
llm_prompt = '''
Assess whether the content of the post delimited by triple backticks aligns with the criteria for clinical depression as defined in DSM-5:
Format your response as a JSON object {'depression':''} with values strictly either 'yes' or 'no'.
Do not return anything else apart from the json object.

Post:
```
'''

In [6]:
input = []
for text in prompt_data["text"].tolist():
    input.append({'prompt': llm_prompt + text + "```"})
print(f"\nSample Input: {input[0]}")


Sample Input: {'prompt': "\nAssess whether the content of the post delimited by triple backticks aligns with the criteria for clinical depression as defined in DSM-5:\nFormat your response as a JSON object {'depression':''} with values strictly either 'yes' or 'no'.\nDo not return anything else apart from the json object.\n\nPost:\n```\nI cut myself for the first time in a year today. and hated that I still loved it. The burning sensation from the blade on my arm and leg. I didnt even cut deep, I barely bled, but the feeling was still almost the same, but without the kick of seeing my own blood trail down my limbs staining the covers in my bed. I told myself that I was only going to cut a few times, but here I am with surely thirty new scars on my body. I fucking hate myself.```"}


In [11]:
prompt_data = hf_dataset.from_list(input)
prompt_data

Dataset({
    features: ['prompt'],
    num_rows: 2872
})

In [10]:
type(input), type(input[0])

(list, dict)